# Unified Dataset — Reference Notebook

Cell-by-cell walkthrough of `ml_service/app/services/build_unified_dataset.py` (bank statement +
SMS merge, recipient-name backfill) and `ml_service/app/services/merchant_normalizer.py` (recipient
canonicalization).

**This notebook imports and runs the real pipeline code — it does not reimplement any logic.**
Both modules live in `ml_service/app/services/` (not here) specifically so they're reusable and
testable outside notebooks; this is the read-only inspection layer on top of them. Several cells
print function source (`inspect.getsource`) so the logic is readable inline without switching
files, but every number below comes from actually running the code.

See `docs/sms_pipeline.md` — "SMS vs. bank-statement overlap" and "Recipient canonicalization" —
for the full narrative behind why this merge exists and what bugs it fixed.

**Data note:** reads real personal financial data (`ml_service/data/captured_sms.csv`,
`ml_preprocessing/CSVS/SpendWise_4yrs_Clean_Merchants.xlsx`) — never commit outputs from this
notebook; both source directories are gitignored.

In [ ]:
import inspect
import os
import sys

import pandas as pd

# Both modules now live in ml_service/app/services/ -- add ml_service/ to sys.path so
# `from app.services...` imports work regardless of where this notebook is launched from.
ML_SERVICE_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', 'ml_service'))
if ML_SERVICE_ROOT not in sys.path:
    sys.path.insert(0, ML_SERVICE_ROOT)

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 200)

import app.services.build_unified_dataset as bud
import app.services.merchant_normalizer as mn

print('Statement source:', bud.WB_PATH)
print('Output path:     ', bud.OUT_PATH)

## Why this merge exists

The SMS pipeline and the bank-statement pipeline are two independent captures of largely the *same*
real-world transactions. A 2026-07-12 audit found **909 of 1,078 SMS financial transactions are
already in the bank statement** (846 by exact reference-id match, 54 by unique date+amount+direction
match, 9 resolved manually) — only **169 are genuinely SMS-only**. Feeding both sources into the same
store without deduping would double-count the 909.

The statement's own `Recipient_Name` field is also truncated to ~8 characters by the export process
(`"PRACHI S"`), while SMS carries the untruncated name (`"PRACHI SAMEER SAWANT"`) for the same
transaction — so the merge does double duty: dedupe, and backfill better names.

## The merge logic (`build()`)

Every SMS financial transaction gets exactly **one** outcome, enforced by an assertion in code (not
just by convention) so a routing bug can never silently duplicate or drop a row:

- `backfilled` — unambiguous statement match, SMS name written into the statement row
- `no_name` — unambiguous statement match, but SMS had no name to offer
- `ambiguous` — confirmed a real statement duplicate (transaction counts agree on both sides), but
  not safely 1:1-pairable by reference id (e.g. two same-amount same-day Airtel recharges) — the
  statement name is left untouched rather than risk a wrong pairing
- `new` — no statement match at all — genuinely SMS-only — becomes a new row

In [ ]:
print(inspect.getsource(bud.build))

### Run the merge

In [ ]:
unified = bud.build()
print('total rows:', len(unified))
print('statement-sourced:', (unified['_source'] == 'statement').sum())
print('sms-only:         ', (unified['_source'] == 'sms_only').sum())
unified.head()

### Spot-check a backfilled row

Truncated statement names should now show the fuller SMS-sourced name.

In [ ]:
backfilled_example = unified[unified['Recipient_Name'].astype(str).str.contains('SAMEER BALIRAM', case=False, na=False)]
backfilled_example[['Transaction_Date', 'Amount', 'Recipient_Name', 'Recipient_Canonical', '_source']].head()

## Recipient canonicalization (`merchant_normalizer.py`)

`build()` recomputes `Recipient_Canonical` from scratch over the whole unified dataset (not patched
incrementally — the old canonical grouping was built on truncated names and would be inconsistent
with the now-fuller `Recipient_Name` column). Three layers, in order:

1. **UPI-ID grouping + fuzzy name clustering** (`normalize_recipients`) — pre-existing two-tier logic.
2. **`merge_prefix_chains`** — safely auto-merges truncation-prefix variants the fuzzy tiers miss.
3. **`MANUAL_ALIASES`** — a small hand-curated map for cases that are unsafe to merge algorithmically
   but safe by real-world knowledge (e.g. `"BHARTI AIRTEL LI"` and `"AIRTEL PREPAID R"` are the same
   telecom company, not a name-prefix relationship).

### The safety guard

A real bug found during this work: two different real people sharing a family UPI handle (`"PRACHI
SAMEER SAWANT"` and `"YASH SAMEER SAWANT"` — different first names, same surname) were being merged
into one canonical name, since fuzzy similarity alone can't tell them apart when they share 2 of 3
tokens and a UPI ID. `_first_names_conflict` fixes this — forces two names with unambiguously
different, non-prefix-related first tokens into different clusters regardless of similarity score.

In [ ]:
print(inspect.getsource(mn._first_names_conflict))

In [ ]:
# Verify the fix directly: these must resolve to DIFFERENT canonical names.
py_check = unified[unified['Recipient_Name'].astype(str).str.contains('PRACHI SAMEER SAWANT|YASH SAMEER SAWANT', case=False, na=False)]
print('Prachi/Yash canonical names (must be 2+ distinct, never merged):')
print(py_check['Recipient_Canonical'].unique())

### Safe truncation-chain merging

`merge_prefix_chains` merges a short truncated name into the single longer name it unambiguously
chains into. If a short form is compatible with two or more genuinely different longer names in the
same component (e.g. two different real people both named `"MAHENDRA ..."`), it's correctly left
unmerged rather than guessed — verified below against real cases found in this dataset.

In [ ]:
print(inspect.getsource(mn.merge_prefix_chains))

In [ ]:
# Sameer Baliram Sawant's ~8 raw spellings should now collapse to one canonical name.
sameer = unified[unified['Recipient_Name'].astype(str).str.contains('SAMEER', case=False, na=False)]
print('Distinct Sameer-related canonical names:', sameer['Recipient_Canonical'].nunique())
print(sorted(sameer['Recipient_Canonical'].dropna().unique()))
print()

# Two different real people who happen to share a name prefix -- must stay separate.
mahendra = unified[unified['Recipient_Name'].astype(str).str.upper().str.startswith('MAHENDRA')]
print('Mahendra canonical names (must be 2+ distinct people, never merged):')
print(mahendra['Recipient_Canonical'].unique())

### Manual aliases (companies / recurring income sources)

The residual cases `merge_prefix_chains` correctly refuses to guess (different word order, not a
prefix relationship) but are safe by real-world knowledge -- never a case of two different people.

In [ ]:
for name, alias in bud.MANUAL_ALIASES.items():
    print(f'{name!r:28} -> {alias!r}')

## Save output

`build()` already wrote `CSVS/SpendWise_Unified_Merchants.xlsx` when run above. Re-run this cell only
if you've modified `unified` in this notebook session and want to persist those changes instead.

In [ ]:
# unified.to_excel(bud.OUT_PATH, index=False)
# print('Wrote', len(unified), 'rows to', bud.OUT_PATH)